# UFLD stage 1: pretrain the causal Transformer

Trains the causal Transformer on random patch sequences with binary cross-entropy and a supervised contrastive loss, then evaluates it on the test set.

In [ ]:
import os
import sys

# Run from the repository root so that `src/`, `data/` and `models/` resolve correctly
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [ ]:
import torch

from src.config import MODEL_CONFIGS, classifier_checkpoint
from src.models.classifier.transformer import CausalTransformerSupCon
from src.trainer.train_transformer import training_loop_transformer
from src.utils.contrastive_loss import SupConLoss
from src.utils.dataloader import load_mixedwm38_splits, get_loaders
from src.utils.utils import seed_everything, accuracy_transformer

## Configuration

In [ ]:
SEED = 4
TRAIN = False          # False: load the checkpoint from models/
MODEL_SIZE = 'large'   # 'small' | 'medium' | 'large' (Table 1)

DATA_PATH = 'data/MixedWM38.npz'

seed_everything(seed=SEED)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

dataset_name = 'mixedwm38'
num_classes = 8
input_channels = 1
img_size = (52, 52)
batch_size = 128

In [ ]:
# Faster training on NVIDIA GPUs
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

## Model

In [ ]:
embedding_dim, num_layers = MODEL_CONFIGS[MODEL_SIZE]
hidden_dim = 2 * embedding_dim
num_heads = 4
patch_size = 4
num_patches = (img_size[0] // patch_size) ** 2   # 169

classifier = CausalTransformerSupCon(
    input_dim=patch_size ** 2,
    hidden_dim=hidden_dim,
    num_classes=num_classes,
    seq_len=num_patches,
    num_layers=num_layers,
    num_heads=num_heads,
    patch_size=patch_size,
    dropout_prob=0.1,
    dataset_name=dataset_name,
    batch_first=True,
    layer_norm=True,
    use_cls_token=False,
    embedding_dim=embedding_dim,
    permute_patches=True,   # train on random patch orders
    use_patch_id_emb=True,
).to(device)

## Data

In [ ]:
train_dataset, val_dataset, calib_dataset, test_dataset = load_mixedwm38_splits(
    DATA_PATH, img_size=img_size, input_channels=input_channels, num_classes=num_classes)

train_loader, test_loader, val_loader, calib_loader = get_loaders(
    train_dataset, test_dataset, val_dataset=val_dataset, calib_dataset=calib_dataset, batch_size=batch_size)

## Training

In [ ]:
lr = 1e-3
wd = 1e-6
c = 0.1                 # weight of the supervised contrastive loss
n_epochs = 240
num_output_heads = 64   # random prefix positions supervised per batch

criterion = torch.nn.BCEWithLogitsLoss(reduction='none')
supcon_loss = SupConLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=lr, weight_decay=wd)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-6)

file_name = classifier_checkpoint(dataset_name, SEED, num_layers, hidden_dim, num_heads=num_heads,
                                  num_output_heads=num_output_heads, c=c)
os.makedirs(os.path.dirname(file_name), exist_ok=True)

if TRAIN:
    classifier, _ = training_loop_transformer(
        model=classifier,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        valid_loader=val_loader,
        epochs=n_epochs,
        num_output_heads=num_output_heads,
        supcon_loss=supcon_loss,
        supcon_weight=c,
        file_name=file_name,
        device=device,
    )

classifier.load_state_dict(torch.load(file_name, map_location=device))

## Evaluation

In [ ]:
num_params = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")

seed_everything(seed=SEED)
prediction_index = 104   # evaluate after 105 acquired patches; use -1 for the full sequence
test_acc, test_f1 = accuracy_transformer(
    classifier, test_loader, device=device, prediction_index=prediction_index)
print(f'Test accuracy: {100 * test_acc:.2f}')
print(f'Test F1:       {test_f1:.3f}')

In [ ]:
# Accuracy as a function of the number of (randomly ordered) acquired patches
steps = list(range(0, num_patches, 10)) + [-1]
for t in steps:
    seed_everything(seed=SEED)
    acc_t, f1_t = accuracy_transformer(classifier, test_loader, device=device, prediction_index=t)
    n_seen = num_patches if t == -1 else t + 1
    print(f"{n_seen:4d} patches  acc = {100 * acc_t:.2f}  F1 = {f1_t:.3f}")